<a href="https://colab.research.google.com/github/zwartepiet-hash/Young-Warrior-Logistics/blob/main/The_Warrior_Translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Telepítés
!pip install -q gradio deep-translator gTTS fpdf openai-whisper librosa soundfile

import gradio as gr
import whisper
import librosa
import soundfile as sf
from deep_translator import GoogleTranslator
from gtts import gTTS
from fpdf import FPDF
from datetime import datetime
import os

# --- 2. MODELL BETÖLTÉSE (High-End: Large-v3) ---
print("A legmagasabb szintű modell betöltése (large-v3)... Ez eltarthat egy percig.")
# Ez a modell már anyanyelvi szintű értést biztosít
model = whisper.load_model("large-v3")
print("Warrior, a 'Large' egység hadrendbe állt!")

# --- 3. JAVÍTOTT FUNKCIÓ (A maximális precízióért) ---
def translate_speech(audio_path, source_lang, target_lang):
    if audio_path is None: return "Nincs hang!", None, ""
    lang_codes = {"Magyar": "hu", "English": "en", "Deutsch": "de"}
    src, dst = lang_codes[source_lang], lang_codes[target_lang]

    audio_res = None

    try:
        audio_data, sr = librosa.load(audio_path, sr=16000)
        fixed_path = "fixed.wav"
        sf.write(fixed_path, audio_data, 16000)

        # Large modellnél a beam_size=5 és a magyar nyelv kényszerítése
        result = model.transcribe(
            fixed_path,
            language="hu" if source_lang == "Magyar" else src,
            task="transcribe",
            beam_size=5
        )

        original = str(result.get("text", "")).strip()
        if not original: return "Nem hallottam semmit...", None, ""

        translated = GoogleTranslator(source=src, target=dst).translate(original)
        tts = gTTS(text=translated, lang=dst)
        audio_res = "result.mp3"
        tts.save(audio_res)

        report = f"[{datetime.now().strftime('%H:%M')}]\n(PRECÍZIÓS MÓD: {source_lang}->{target_lang})\nEREDETI: {original}\nFORDÍTÁS: {translated}\n"
        return translated, audio_res, report
    except Exception as e:
        return f"Hiba: {str(e)}", None, ""

def clean_text_for_pdf(text):
    # Brutálisan tiszta karakterkezelés a PDF-hez
    replacements = {
        'ő': 'ö', 'ű': 'ü', 'Ő': 'Ö', 'Ű': 'Ü',
        '’': "'", '–': '-', '„': '"', '”': '"'
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    # Csak latin-1 kompatibilis karaktereket hagyunk meg
    return text.encode('latin-1', 'replace').decode('latin-1')

def save_to_pdf(history):
    if not history: return None
    try:
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", size=12) # A Helvetica stabilabb

        # Cím
        pdf.set_font("Helvetica", "B", 16)
        pdf.cell(0, 10, txt="WARRIOR TRANSLATION LOG", ln=True, align='C')
        pdf.ln(10)

        # Tartalom sortöréssel (multi_cell a titok!)
        pdf.set_font("Helvetica", size=11)
        clean_history = clean_text_for_pdf(history)

        # A 0 szélesség azt jelenti, hogy a margóig tart, a 10 a sor magassága
        pdf.multi_cell(0, 10, txt=clean_history)

        fname = "Warrior_Log_v3.pdf"
        pdf.output(fname)
        return fname
    except Exception as e:
        print(f"PDF Hiba: {e}")
        return None

# --- 4. UI (INTERNATIONALLY READY) ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ Warrior Translator v4.1 - International Edition")

    with gr.Row():
        with gr.Column():
            src_lang = gr.Dropdown(
                choices=["Magyar", "English", "Deutsch"],
                value="Magyar",
                label="Source Language / Forrásnyelv"
            )
            audio_in = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="🎤 Record Audio / Hang rögzítése"
            )
            target_lang = gr.Dropdown(
                choices=["Magyar", "English", "Deutsch"],
                value="English",
                label="Target Language / Célnyelv"
            )
            start_btn = gr.Button("⚔️ START TRANSLATION / FORDÍTÁS INDÍTÁSA", variant="primary")

        with gr.Column():
            text_out = gr.Textbox(label="Translated Text / Lefordított szöveg")
            audio_out = gr.Audio(label="🔈 AI Voice / Hanggenerálás")
            hist_out = gr.Textbox(label="📜 History Log / Előzmények", lines=8)
            pdf_btn = gr.Button("📄 EXPORT PDF / PDF MENTÉSE")
            pdf_file = gr.File(label="Download File / Letöltés")

    # Kapcsolatok
    start_btn.click(translate_speech, [audio_in, src_lang, target_lang], [text_out, audio_out, hist_out])
    pdf_btn.click(save_to_pdf, hist_out, pdf_file)

# Start!
demo.launch(share=True)



  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 25.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.23.1 which is incompatible.
A legmagasabb szintű modell betöltése (large-v3)... Ez eltarthat egy percig.


100%|█████████████████████████████████████| 2.88G/2.88G [00:35<00:00, 86.8MiB/s]


Warrior, a 'Large' egység hadrendbe állt!


/tmp/ipykernel_783/3670682762.py:92: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://58ff91d4a1968867a2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
